In [ ]:

import csv
import logging
import math
import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm


def set_seed(seed: int = 42):
    """Lock down all randomness sources for reproducibility."""
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False




def get_dataloaders(batch_size: int = 128):
    """
    Downloads CIFAR-10 and returns train / val / test loaders.
    Train gets augmentation (flip + crop); val/test get only normalization.
    80/20 split from the official 50k training set.
    """
    train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2023, 0.1994, 0.2010))
    ])
    eval_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2023, 0.1994, 0.2010))
    ])

    full_train = torchvision.datasets.CIFAR10(
        root='./data', train=True, download=True, transform=train_transform)
    test_set = torchvision.datasets.CIFAR10(
        root='./data', train=False, download=True, transform=eval_transform)

    train_size = int(0.8 * len(full_train))
    val_size   = len(full_train) - train_size
    train_set, val_set = random_split(full_train, [train_size, val_size])

    train_loader = DataLoader(train_set, batch_size=batch_size,
                              shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_set,   batch_size=batch_size,
                              shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_set,  batch_size=batch_size,
                              shuffle=False, num_workers=2, pin_memory=True)

    return train_loader, val_loader, test_loader



class PrunableLinear(nn.Module):
    """
    Drop-in replacement for nn.Linear that learns to prune its own weights.

    Each weight w_ij gets a companion gate_score g_ij (same tensor shape).
    During the forward pass:
        gates        = sigmoid(gate_scores × temperature)   ∈ (0, 1)
        pruned_weight = weight × gates                      (element-wise)
        output        = x @ pruned_weight.T + bias

    When a gate → 0, its weight contributes nothing → effectively pruned.
    Gradients flow through both weight and gate_scores via autograd.

    Key init decision:
        gate_scores initialized to +2.0
        → sigmoid(2.0) ≈ 0.88  → gates start mostly OPEN
        L1 pressure then drives weak gates to exactly 0.
        This gives the desired bimodal distribution (spike at 0, cluster near 1).
    """

    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.in_features  = in_features
        self.out_features = out_features

        # Standard learnable weight matrix and bias
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self.bias   = nn.Parameter(torch.empty(out_features))

        # One gate_score per weight — SAME shape, separate parameter
        # Init 0.0 → sigmoid(0) = 0.5 → gates start at midpoint
        # L1 pressure can push weak gates to 0 even at small lambda values
        self.gate_scores = nn.Parameter(
            torch.full((out_features, in_features), 2.0)
        )

        self._reset_parameters()

    def _reset_parameters(self):
        """Kaiming init for weights (standard practice for ReLU nets)."""
        nn.init.kaiming_normal_(self.weight, a=math.sqrt(5))
        fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight)
        bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
        nn.init.uniform_(self.bias, -bound, bound)
        # gate_scores already zero-initialized in __init__

    def forward(self, x: torch.Tensor, temperature: float = 1.0) -> torch.Tensor:
        """
        temperature: controls sharpness of sigmoid.
            Low T  (early training) → soft gates, smooth gradients
            High T (late training)  → hard binary-like gates, clear pruning
        """
        gates         = torch.sigmoid(self.gate_scores * temperature)
        pruned_weight = self.weight * gates          # element-wise mask
        return F.linear(x, pruned_weight, self.bias)

    def get_gates(self, temperature: float = 1.0) -> torch.Tensor:
        """Returns gate values (detached) for analysis."""
        with torch.no_grad():
            return torch.sigmoid(self.gate_scores * temperature)

    def get_sparsity(self, threshold: float = 1e-2,
                     temperature: float = 1.0) -> dict:
        """Per-layer sparsity stats."""
        gates = self.get_gates(temperature)
        n     = gates.numel()
        return {
            'sparsity_pct'  : (gates < threshold).sum().item() / n * 100,
            'mean_gate'     : gates.mean().item(),
            'pct_near_zero' : (gates < 1e-3).sum().item() / n * 100,
            'pct_near_one'  : (gates > 0.99).sum().item() / n * 100,
            'num_weights'   : n,
        }




class SelfPruningCNN(nn.Module):
    """
    CNN backbone with two PrunableLinear classifiers.

    Feature extractor:  Conv(3→32) → ReLU → MaxPool
                        Conv(32→64) → ReLU → MaxPool
                        → output shape: (B, 64, 8, 8) → flatten to 4096

    Classifier:         PrunableLinear(4096 → 512) → ReLU
                        PrunableLinear(512  → 10)
    """

    def __init__(self):
        super().__init__()

        # CNN feature extractor — standard layers, NOT prunable
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        # 32×32 → pool → 16×16 → pool → 8×8; 64 channels → 64*8*8 = 4096
        self.flatten = nn.Flatten()

        # PrunableLinear replaces nn.Linear — gates live here
        self.fc1  = PrunableLinear(64 * 8 * 8, 512)
        self.relu = nn.ReLU(inplace=True)
        self.fc2  = PrunableLinear(512, 10)

    def forward(self, x: torch.Tensor, temperature: float = 1.0) -> torch.Tensor:
        x = self.features(x)
        x = self.flatten(x)
        x = self.fc1(x, temperature)
        x = self.relu(x)
        x = self.fc2(x, temperature)
        return x

    # ── Sparsity helpers ──────────────────────────────────────

    def get_sparsity_loss(self, temperature: float = 1.0) -> torch.Tensor:
        """
        L1 norm of all gate values, normalized by total gate count.
        Normalized so lambda is comparable across different architectures.

        SparsityLoss = (1/N) * Σ sigmoid(gate_scores × T)

        Why L1 encourages sparsity:
            L1 applies a constant gradient (-λ sign(gate)) regardless of magnitude.
            Unlike L2 which slows down as values approach 0, L1 keeps pushing
            with equal force all the way to 0 → gates collapse to exactly 0.
        """
        total_loss  = torch.tensor(0.0, device=next(self.parameters()).device)
        total_gates = 0

        for m in self.modules():
            if isinstance(m, PrunableLinear):
                gates        = torch.sigmoid(m.gate_scores * temperature)
                total_loss  += gates.sum()
                total_gates += gates.numel()

        return total_loss / total_gates if total_gates > 0 else total_loss

    def get_network_sparsity(self, threshold: float = 1e-2,
                             temperature: float = 1.0) -> dict:
        """Overall and per-layer sparsity statistics."""
        all_gates   = []
        layer_stats = {}
        idx         = 1

        for m in self.modules():
            if isinstance(m, PrunableLinear):
                stats               = m.get_sparsity(threshold, temperature)
                layer_stats[f'l{idx}'] = stats
                all_gates.append(m.get_gates(temperature).cpu().flatten())
                idx += 1

        all_gates_cat = torch.cat(all_gates)
        n             = all_gates_cat.numel()

        return {
            'overall_sparsity' : (all_gates_cat < threshold).sum().item() / n * 100,
            'overall_mean_gate': all_gates_cat.mean().item(),
            'pct_near_zero'    : (all_gates_cat < 1e-3).sum().item() / n * 100,
            'pct_near_one'     : (all_gates_cat > 0.99).sum().item() / n * 100,
            'layer_stats'      : layer_stats,
            'all_gates'        : all_gates_cat.numpy(),
        }




def get_temperature(epoch: int, total_epochs: int,
                    t_start: float = 1.0, t_end: float = 5.0) -> float:
    """
    Linearly anneals temperature from t_start → t_end over training.

    Low T  → sigmoid is soft → smooth gradients early on
    High T → sigmoid sharpens toward step function → gates commit to 0 or 1

    This is the key improvement over the baseline's fixed temperature.
    """
    return t_start + (t_end - t_start) * (epoch / max(total_epochs - 1, 1))




def train_one_epoch(model, loader, criterion, optimizer,
                    lam: float, temperature: float, device):
    model.train()
    total_loss = cls_total = spar_total = 0.0
    correct = n = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()

        logits   = model(imgs, temperature)
        cls_loss = criterion(logits, labels)
        spar_loss = model.get_sparsity_loss(temperature)
        loss     = cls_loss + lam * spar_loss

        loss.backward()
        optimizer.step()

        bs           = imgs.size(0)
        total_loss  += loss.item() * bs
        cls_total   += cls_loss.item() * bs
        spar_total  += spar_loss.item() * bs
        correct     += logits.argmax(1).eq(labels).sum().item()
        n           += bs

    return total_loss/n, cls_total/n, spar_total/n, 100*correct/n


@torch.no_grad()
def evaluate(model, loader, criterion, temperature: float, device):
    model.eval()
    total_loss = correct = n = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits      = model(imgs, temperature)
        total_loss += criterion(logits, labels).item() * imgs.size(0)
        correct    += logits.argmax(1).eq(labels).sum().item()
        n          += imgs.size(0)

    return total_loss/n, 100*correct/n




def run_experiment(lam: float, epochs: int,
                   train_loader, val_loader, test_loader, device):
    logging.info(f"\n{'='*55}")
    logging.info(f"  Experiment: λ = {lam}  |  {epochs} epochs")
    logging.info(f"{'='*55}")

    model     = SelfPruningCNN().to(device)
    criterion = nn.CrossEntropyLoss()

    # Separate param groups: weight decay only on conv/fc weights, NOT gates
    # If weight decay hits gate_scores it competes with L1 → noisy pruning
    weight_params = [p for n, p in model.named_parameters()
                     if 'gate_scores' not in n]
    gate_params   = [p for n, p in model.named_parameters()
                     if 'gate_scores' in n]

    optimizer = optim.Adam([
        {'params': weight_params, 'lr': 1e-3, 'weight_decay': 1e-4},
        {'params': gate_params,   'lr': 5e-3, 'weight_decay': 0.0},  # faster LR for gates
    ])
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)

    history = {k: [] for k in [
        'train_loss', 'val_loss', 'train_acc', 'val_acc',
        'cls_loss', 'spar_loss', 'sparsity', 'mean_gate',
        'l1_sparsity', 'l2_sparsity', 'temperature'
    ]}

    best_val_acc      = 0.0
    pruning_onset     = 'N/A'
    best_model_path   = f'results/best_model_lam{lam}.pth'
    start             = time.time()

    for epoch in tqdm(range(epochs), desc=f"λ={lam}"):
        T = get_temperature(epoch, epochs)

        t_loss, t_cls, t_spar, t_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, lam, T, device)
        v_loss, v_acc = evaluate(
            model, val_loader, criterion, T, device)
        scheduler.step()

        net_stats = model.get_network_sparsity(temperature=T)
        spar      = net_stats['overall_sparsity']
        ls        = net_stats['layer_stats']

        if pruning_onset == 'N/A' and spar > 5.0:
            pruning_onset = epoch + 1

        history['train_loss'].append(t_loss)
        history['val_loss'].append(v_loss)
        history['train_acc'].append(t_acc)
        history['val_acc'].append(v_acc)
        history['cls_loss'].append(t_cls)
        history['spar_loss'].append(t_spar)
        history['sparsity'].append(spar)
        history['mean_gate'].append(net_stats['overall_mean_gate'])
        history['l1_sparsity'].append(ls.get('l1', {}).get('sparsity_pct', 0))
        history['l2_sparsity'].append(ls.get('l2', {}).get('sparsity_pct', 0))
        history['temperature'].append(T)

        if v_acc > best_val_acc:
            best_val_acc = v_acc
            torch.save(model.state_dict(), best_model_path)

        if (epoch + 1) % 10 == 0:
            logging.info(
                f"  Epoch {epoch+1:3d}/{epochs} | T={T:.1f} | "
                f"TrainAcc={t_acc:.1f}% | ValAcc={v_acc:.1f}% | "
                f"Sparsity={spar:.1f}%"
            )

    elapsed = time.time() - start

    # ── Reload best checkpoint for final eval ─────────────────
    model.load_state_dict(torch.load(best_model_path,
                                     map_location=device, weights_only=True))
    final_T         = get_temperature(epochs - 1, epochs)
    _, test_acc     = evaluate(model, test_loader, criterion, final_T, device)
    final_stats     = model.get_network_sparsity(temperature=final_T)
    ls_final        = final_stats['layer_stats']

    # ── Plots ─────────────────────────────────────────────────
    plot_training_curves(history, lam)
    plot_gate_distribution(final_stats['all_gates'], lam)

    logging.info(f"\n  ✅ λ={lam} done | "
                 f"TestAcc={test_acc:.2f}% | "
                 f"Sparsity={final_stats['overall_sparsity']:.2f}%")

    return {
        'Lambda'            : lam,
        'Test Accuracy'     : round(test_acc, 2),
        'Val Accuracy'      : round(best_val_acc, 2),
        'Overall Sparsity'  : round(final_stats['overall_sparsity'], 2),
        'Mean Gate'         : round(final_stats['overall_mean_gate'], 4),
        'Pct Near Zero'     : round(final_stats['pct_near_zero'], 2),
        'Pct Near One'      : round(final_stats['pct_near_one'], 2),
        'L1 Sparsity'       : round(ls_final.get('l1', {}).get('sparsity_pct', 0), 2),
        'L2 Sparsity'       : round(ls_final.get('l2', {}).get('sparsity_pct', 0), 2),
        'Pruning Onset'     : pruning_onset,
        'Training Time (s)' : round(elapsed, 1),
    }


def plot_training_curves(history: dict, lam: float):
    """6-panel training dashboard for a single lambda run."""
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    fig.suptitle(f'Training Dashboard — λ={lam}', fontsize=14, fontweight='bold')

    # Loss
    ax = axes[0, 0]
    ax.plot(epochs, history['train_loss'], label='Train')
    ax.plot(epochs, history['val_loss'],   label='Val')
    ax.set_title('Total Loss'); ax.set_xlabel('Epoch')
    ax.legend()

    # Accuracy
    ax = axes[0, 1]
    ax.plot(epochs, history['train_acc'], label='Train')
    ax.plot(epochs, history['val_acc'],   label='Val')
    ax.set_title('Accuracy (%)'); ax.set_xlabel('Epoch')
    ax.legend()

    # Sparsity over epochs
    ax = axes[0, 2]
    ax.plot(epochs, history['sparsity'], color='darkorange')
    ax.set_title('Overall Sparsity (%)'); ax.set_xlabel('Epoch')

    # CE vs Sparsity loss scatter
    ax = axes[1, 0]
    sc = ax.scatter(history['cls_loss'], history['spar_loss'],
                    c=list(epochs), cmap='viridis', s=20)
    fig.colorbar(sc, ax=ax, label='Epoch')
    ax.set_xlabel('CE Loss'); ax.set_ylabel('Sparsity Loss')
    ax.set_title('CE vs Sparsity Loss')

    # Layer-wise sparsity
    ax = axes[1, 1]
    ax.plot(epochs, history['l1_sparsity'], label='FC1')
    ax.plot(epochs, history['l2_sparsity'], label='FC2')
    ax.set_title('Layer-wise Sparsity (%)'); ax.set_xlabel('Epoch')
    ax.legend()

    # Mean gate + temperature
    ax = axes[1, 2]
    ax2 = ax.twinx()
    ax.plot(epochs, history['mean_gate'],   color='steelblue', label='Mean Gate')
    ax2.plot(epochs, history['temperature'], color='tomato',
             linestyle='--', label='Temperature')
    ax.set_title('Mean Gate Value & Temperature')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Mean Gate', color='steelblue')
    ax2.set_ylabel('Temperature', color='tomato')
    ax.legend(loc='upper left'); ax2.legend(loc='upper right')

    plt.tight_layout()
    plt.savefig(f'results/plots/training_lam{lam}.png', dpi=120)
    plt.close()


def plot_gate_distribution(all_gates: np.ndarray, lam: float):
    """
    Histogram of final gate values.
    A successful pruning shows: large spike at 0, cluster near 1.
    """
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(all_gates, bins=100, color='steelblue',
            edgecolor='white', linewidth=0.3)
    ax.axvline(x=0.01,  color='red',    linestyle='--',
               linewidth=1.5, label='Prune threshold (0.01)')
    ax.axvline(x=0.99,  color='green',  linestyle='--',
               linewidth=1.5, label='Active threshold (0.99)')
    ax.set_title(f'Final Gate Value Distribution — λ={lam}',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Gate Value  (0 = pruned, 1 = fully active)')
    ax.set_ylabel('Count')
    ax.legend()

    near_zero = (all_gates < 0.01).mean() * 100
    near_one  = (all_gates > 0.99).mean() * 100
    ax.text(0.5, 0.92,
            f'Near-zero: {near_zero:.1f}%   Near-one: {near_one:.1f}%',
            transform=ax.transAxes, ha='center',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    plt.tight_layout()
    plt.savefig(f'results/plots/gate_dist_lam{lam}.png', dpi=120)
    plt.close()


def plot_cross_lambda(results: list, csv_path: str):
    """Cross-lambda comparison plots (4 panels)."""
    df = pd.DataFrame(results)

    fig, axes = plt.subplots(2, 2, figsize=(13, 10))
    fig.suptitle('Cross-Lambda Comparison', fontsize=14, fontweight='bold')

    lams = df['Lambda'].tolist()

    # Accuracy vs Lambda
    ax = axes[0, 0]
    ax.plot(lams, df['Test Accuracy'], marker='o', label='Test')
    ax.plot(lams, df['Val Accuracy'],  marker='s', label='Val')
    ax.set_xscale('log'); ax.set_title('Accuracy vs Lambda')
    ax.set_xlabel('Lambda'); ax.set_ylabel('Accuracy (%)')
    ax.legend()

    # Sparsity vs Lambda
    ax = axes[0, 1]
    ax.plot(lams, df['Overall Sparsity'], marker='o', color='darkorange')
    ax.set_xscale('log'); ax.set_title('Sparsity vs Lambda')
    ax.set_xlabel('Lambda'); ax.set_ylabel('Sparsity (%)')

    # Accuracy–Sparsity trade-off
    ax = axes[1, 0]
    ax.plot(df['Overall Sparsity'], df['Test Accuracy'], marker='o')
    for _, row in df.iterrows():
        ax.annotate(f"λ={row['Lambda']}",
                    (row['Overall Sparsity'], row['Test Accuracy']),
                    textcoords='offset points', xytext=(5, 5))
    ax.set_title('Accuracy vs Sparsity Trade-off')
    ax.set_xlabel('Sparsity (%)'); ax.set_ylabel('Test Accuracy (%)')

    # Layer-wise sparsity bar chart
    ax = axes[1, 1]
    x     = np.arange(len(lams))
    width = 0.35
    ax.bar(x - width/2, df['L1 Sparsity'], width, label='FC1')
    ax.bar(x + width/2, df['L2 Sparsity'], width, label='FC2')
    ax.set_xticks(x); ax.set_xticklabels([f'λ={l}' for l in lams])
    ax.set_title('Layer-wise Sparsity'); ax.set_ylabel('Sparsity (%)')
    ax.legend()

    plt.tight_layout()
    plt.savefig('results/plots/cross_lambda.png', dpi=120)
    plt.close()




def main():
    set_seed(42)
    os.makedirs('results/plots', exist_ok=True)

    logging.basicConfig(level=logging.INFO,
                        format='%(levelname)s: %(message)s')

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    logging.info(f"Device: {device}")

    train_loader, val_loader, test_loader = get_dataloaders(batch_size=128)


    lambdas = [0.001, 0.01, 0.1]
    epochs  = 50

    results = []
    for lam in lambdas:
        res = run_experiment(lam, epochs,
                             train_loader, val_loader, test_loader, device)
        results.append(res)

    # ── Save CSV results table ─────────────────────────────────
    csv_path = 'results/results_table.csv'
    fields   = list(results[0].keys())
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(results)

    # ── Cross-lambda plots ─────────────────────────────────────
    plot_cross_lambda(results, csv_path)

    # ── Print final summary table ──────────────────────────────
    print("\n" + "="*65)
    print(f"{'Lambda':>10} | {'Test Acc':>10} | {'Sparsity':>10} | {'Near-Zero%':>12}")
    print("-"*65)
    for r in results:
        print(f"{r['Lambda']:>10} | {r['Test Accuracy']:>9.2f}% | "
              f"{r['Overall Sparsity']:>9.2f}% | {r['Pct Near Zero']:>11.2f}%")
    print("="*65)
    logging.info("Done! All results saved in results/")


if __name__ == '__main__':
    main()

INFO: Device: cuda


Files already downloaded and verified
Files already downloaded and verified


INFO: 
INFO:   Experiment: λ = 0.001  |  50 epochs
INFO: =======================================================
λ=0.001: 100%|██████████| 50/50 [18:26<00:00, 22.12s/it]
INFO: 
  ✅ λ=0.001 done | TestAcc=81.98% | Sparsity=0.00%
INFO: 
INFO:   Experiment: λ = 0.01  |  50 epochs
INFO: =======================================================
λ=0.01: 100%|██████████| 50/50 [22:05<00:00, 26.51s/it]
INFO: 
  ✅ λ=0.01 done | TestAcc=81.27% | Sparsity=11.36%
INFO: 
INFO:   Experiment: λ = 0.1  |  50 epochs
INFO: =======================================================
λ=0.1: 100%|██████████| 50/50 [18:52<00:00, 22.64s/it]
INFO: 
  ✅ λ=0.1 done | TestAcc=81.18% | Sparsity=82.12%
INFO: Done! All results saved in results/



    Lambda |   Test Acc |   Sparsity |   Near-Zero%
-----------------------------------------------------------------
     0.001 |     81.98% |      0.00% |        0.00%
      0.01 |     81.27% |     11.36% |        0.00%
       0.1 |     81.18% |     82.12% |       79.81%
